In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.variable_profiling import eda_per_table_printing_results
from default_risk.scripts.variable_profiling import eda_per_table_persisting_result_html
from default_risk.scripts.variable_profiling import create_files_nulls_per_colmun
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_series_given_rows
from default_risk.scripts.auxiliar_eda_function import recreate_and_sort_the_serie_given_ids
from default_risk.scripts.auxiliar_eda_function import check_invariant
import default_risk.config as cfg
import logging
import dtale
import dtale.global_state as dtale_global
import gc

dtale_global.cleanup()
gc.collect()

credit_card_df= pd.read_csv(cfg.CREDIT_CARD_BALANCE)

data_frame_size=len(credit_card_df)

credit_card_df.sort_values(["SK_ID_PREV","MONTHS_BALANCE"],inplace=True)


log = logging.getLogger('werkzeug')

with open(cfg.SCHEMA_JSON, "r") as f:
    schema = json.load(f)


#auxiliar functions

def full_sorted_series_id(ids : pd.Series) -> pd.DataFrame :
    return recreate_and_sort_the_serie_given_ids(ids,credit_card_df,"SK_ID_PREV","MONTHS_BALANCE")
     

def full_sorted_series_row(rows : pd.DataFrame) -> pd.DataFrame :
    return recreate_and_sort_series_given_rows(rows,credit_card_df,"SK_ID_PREV","MONTHS_BALANCE")    
    



Invariants found at the moment: (# number of cell with the proofs and relevant code)

1- If CNT_INSTALMENT_MATURE_CUM > 0 then AMT_INST_MIN_REGULARITY is also !=0. #8

2- Count installment monotonicity: CNT_INSTALMENT_MATURE_CUM is monotonically non-decreasing over time (MONTHS_BALANCE). Starts increasing when AMT_INST_MIN_REGULARITY becomes > 0 and stops when AMT_BALANCE = 0
or in a more formal definition, if Xt-1[CNT_INSTALMENT_MATURE_CUM] > 0 (we are not at the beginning of the time series) and Xt-1[CNT_INSTALMENT_MATURE_CUM] ==Xt[CNT_INSTALMENT_MATURE_CUM] then Xt-1[AMT_BALANCE] == 0  (100%) #8, #9, #10, #11

3- Once the loan reaches the "Completed" status, the subsequent records of the sequence also have the same status. (100%) #18

4- Magnitude Hierarchy: AMT_BALANCE > AMT_RECEIVABLE_PRINCIPAL (100%) #3

5- Gross vs net: AMT_TOTAL_RECEIVABLE >= AMT_RECEIVABLE  (100%) #3





Soft constraints:

3- AMT_TOTAL_RECEIVABLE >= AMT_RECEIVABLE_PRINCIPAL (97,21%) #3
anomalies: Almost always overpayment (AMT_TOTAL_RECEIVABLE < 0) but the capital of the debt was not updated (AMT_RECEIVABLE_PRINCIPAL > 0) the remaining 550 cases with irregular behavior.

4- AMT_BALANCE & AMT_RECEIVABLE_PRINCIPAL > 0 (99.04%) #3
anomalies: Overpayment

5- AMT_DRAWINGS_CURRENT >= (AMT_DRAWINGS_ATM_CURRENT + AMT_DRAWINGS_POS_CURRENT +  AMT_DRAWINGS_OTHER_CURRENT) (99.94%)  #13
anomalies: Data corruption

6- AMT_TOTAL_RECEIVABLE >= AMT_INST_MIN_REGULARITY  (94,2%) #13
anomalies: AMT_TOTAL_RECEIVABLE < 0 in a month with AMT_INST_MIN_REGULARITY defined. The remaining  cases are penalties in the debt. 



Decisions summary: 

1- Compute the difference between  AMT_TOTAL_RECEIVABLE and AMT_BALANCE to keep it as feature and apply Symlog transformation, drop AMT_TOTAL_RECIVABLE and keep AMT_BALANCE as anchor. #4

2- Compute the difference between AMT_PAYMENT_CURRENT and AMT_PAYMENT_TOTAL_CURRENT, keep it as feature and drop AMT_PAYMENT_CURRENT. #5

4- Treat CNT_INSTALMENT_MATURE_CUM missing values as 0. #10

5- Treat AMT_DRAWINGS_POS_CURRENT, AMT_DRAWINGS_OTHER_CURRENT and AMT_DRAWINGS_ATM_CURRENT missing values as 0. #12

6- Clip AMT_BALANCE at 0 (left tail) and extract the negative values as a flag "have_negative_balance". #3 #16

7- Use the first instant where AMT_INST_MIN_REGULARITY changes from 0 as the first month where the payment is expected. #8

8- if AMT_INST_MIN_REGULARITY > AMT_TOTAL_RECEIVABLE (invariant 6 violated) and AMT_TOTAL_RECEIVABLE > 0, compute "inconsistency_gap" as AMT_INST_MIN_REGULARITY -  AMT_TOTAL_RECEIVABLE and apply log1p transform to normalize the scale (heavy outliers) and highly zero-inflated. #15 #16

9- We define non-closed loans as loans without any record with "Completed" status, with the following subcategories:
    a- Ongoing: the most recent record has MONTHS_BALANCE > -3
    B- Incomplete sequence: The most recent record has MONTHS_BALANCE <= -3

In [ ]:
#files for the data dictionary
create_files_nulls_per_colmun(credit_card_df,"credit_card_balance")

In [ ]:
#run the screening script on credit_card_balance
eda_per_table_printing_results(credit_card_df, schema, "credit_card_balance",False)

In [ ]:
#3

#In order to compare magnitudes, his semantic meanings and how reliavable are in this dataset we proceed to this validations:


check_invariant((credit_card_df["AMT_RECIVABLE"] > credit_card_df["AMT_TOTAL_RECEIVABLE"]), "AMT_RECIVABLE are bigger than AMT_TOTAL_RECEIVABLE", data_frame_size)

check_invariant((credit_card_df["AMT_RECEIVABLE_PRINCIPAL"] > credit_card_df["AMT_TOTAL_RECEIVABLE"]) & ( credit_card_df["CNT_INSTALMENT_MATURE_CUM"] > 1),"AMT_RECEIVABLE_PRINCIPAL are bigger than AMT_TOTAL_RECEIVABLE (once when the payment is ongoing)" , data_frame_size)

check_invariant(credit_card_df["AMT_RECEIVABLE_PRINCIPAL"] > credit_card_df["AMT_BALANCE"], "AMT_RECEIVABLE_PRINCIPAL is bigger than AMT_BALANCE", data_frame_size)

check_invariant((credit_card_df["AMT_BALANCE"] < 0),"AMT_BALANCE is negative", data_frame_size)



In [ ]:
#4
#we identify a high degree of collinearity between AMT_BALANCE and AMT_TOTAL_RECEIVABLE. 
diff = credit_card_df["AMT_BALANCE"] - credit_card_df["AMT_TOTAL_RECEIVABLE"]
non_cero_ammount= (diff != 0).sum()
print("this 2 columns are different in: " + str((non_cero_ammount * 100) / len(diff)) + "% the observations")
diff.describe()
#With 84% of the observations having this 2 columns with equal values but the difference carrying potential signal we decides to:
#1- Drop AMT_TOTAL_RECIVABLE to avoid redundance
#2- Create a feature of the difference
#3- Apply Symlog transformation to the new feature based on the metrics of the .describe() exhibit a distribution dominated by outliers.

In [ ]:
#5
#In the same line, we explore the colineality between "AMT_PAYMENT_CURRENT" & AMT_PAYMENT_TOTAL_CURRENT"
diff = credit_card_df["AMT_PAYMENT_CURRENT"] - credit_card_df["AMT_PAYMENT_TOTAL_CURRENT"]
non_cero_ammount= (diff != 0).sum()
print("this 2 columns are different in: " + str((non_cero_ammount * 100) / len(diff)) + "% the observations")
diff.describe()
#Thes are not redundant but are mathematically linked. Extra_payed_costs = AMT_PAYMENT_TOTAL_CURRENT - AMT_PAYMENT_CURRENT.
#Therefore in order to explicit relationships hard to discover for the model and avoid colineality we will:
#1- Calculate the diference and save in a new column
#2- Drop AMT_PAYMENT_CURRENT because we consider it less reliable and have (19% of missing values) and keep AMT_PAYMENT_TOTAL_CURRENT as anchor (has no missing values)

In [12]:
#6
first_ids_prev_app= (credit_card_df["SK_ID_PREV"].unique())[:100]
dtale.show(full_sorted_series_id(first_ids_prev_app))                                                                                                                                                                                                                                                                                                    

2026-05-15 22:10:39,597 - ERROR    - Exception occurred while processing request: object of type 'NoneType' has no len()
Traceback (most recent call last):
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 120, in _handle_exceptions
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1573, in get_processes
    [_load_process(data_id) for data_id in global_state.keys()],
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1573, in <listcomp>
    [_load_process(data_id) for data_id in global_state.keys()],
     ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kuroc\OneDrive\Escritorio\default risk\default-risk\Lib\site-packages\dtale\views.py", line 1558, in _load_process
    rows=len(data),
       

In [ ]:
#7
#This proof the existence of rows where the minimun ammout to pay that month to avoid delinquency is missing but exist a debt.
rows_with_null= credit_card_df[credit_card_df["AMT_INST_MIN_REGULARITY"].isna()]
mask_exist_debt= rows_with_null["AMT_BALANCE"].notnull() & rows_with_null["AMT_BALANCE"] != 0
rows_with_null_and_debt= rows_with_null[mask_exist_debt]
dtale.show(full_sorted_series_row(rows_with_null_and_debt))
#this pattern tends to show in the first moment the serie represent a debt, and no necessarily are cases of delinquency.

In [ ]:
#8
rows_with_null= credit_card_df[credit_card_df["AMT_INST_MIN_REGULARITY"].isna()]
mask_exist_deb_and_payed_instalments= (rows_with_null["AMT_BALANCE"].notnull()) & (rows_with_null["AMT_BALANCE"] != 0) & (rows_with_null["CNT_INSTALMENT_MATURE_CUM"] > 0)
rows_with_null_and_debt= rows_with_null[mask_exist_deb_and_payed_instalments]
full_sorted_series_row(rows_with_null_and_debt).head()
#this represent that once the client start to repair the debt, always have definied the minimun ammout to avoid delincuency defined 

In [ ]:
#9
#in order to check for the behaivor of nulls and if are legit missing values or corrupted data, we let's continue with "CNT_INSTALMENT_MATURE_CUM".
rows_with_null_values = credit_card_df[credit_card_df["CNT_INSTALMENT_MATURE_CUM"].isna()]
dtale.show(full_sorted_series_row(rows_with_null_values))
#this give us the strong hipotesis that "CNT_INSTALMENT_MATURE_CUM" missing values are "silents ceros".

In [ ]:
#10
#In order to test the hipotesis of "CNT_INSTALMENT_MATURE_CUM" nulls being "silents ceros"
def is_valid_next(series):
    return series.isin([0, 1]) | series.isna()

credit_card_df["NEXT_VALUE_INSTALMENT"] = credit_card_df.groupby("SK_ID_PREV")["CNT_INSTALMENT_MATURE_CUM"].shift(-1)
mask_for_nulls_pattern= (credit_card_df["CNT_INSTALMENT_MATURE_CUM"].isnull()) & ~(is_valid_next(credit_card_df["NEXT_VALUE_INSTALMENT"]))
full_sorted_series_row(credit_card_df[mask_for_nulls_pattern]).head()
#this proof the hipotesis that "CNT_INSTALMENT_MATURE_CUM" missing values are just "silents ceros"


In [ ]:
#11
#We discover a relatioship between "CNT_INSTALMENT_MATURE_CUM" and "AMT_BALANCE", now we want to visualize if there is exception to this.
credit_card_df["CNT_INSTALMENT_IS_CONSTANT_VS_PREV"] = (
    credit_card_df.groupby("SK_ID_PREV")["CNT_INSTALMENT_MATURE_CUM"]
      .diff() == 0
)
rows_without_invariant_fulfiled = credit_card_df[(credit_card_df["CNT_INSTALMENT_IS_CONSTANT_VS_PREV"] == True) & (credit_card_df["AMT_BALANCE"] > 0)]
dtale.show(full_sorted_series_row(rows_without_invariant_fulfiled))
#Seems like the debt is created can have "delay" util start to be paied. But is easy to detect because if we change "AMT_BALANCE" for "AMT_INST_MIN_REGULARITY" > 0 don't exist cases where this condition
#Is true. So the time of the counter of "CNT_INSTALMENT_MATURE_CUM" start to run in the moment where the minimun ammout to don't incurry in delincuency is setted, and stop in the moment "AMT_BALANCE" hits 0.


In [ ]:
#12
#To check if the nulls at AMT_DRAWINGS_ATM_CURRENT, AMT_DRAWINGS_POS_CURRENT and AMT_DRAWINGS_OTHER_CURRENT are just silent ceros we will analize the correlation with AMT_DRAWINGS_CURRENT
check_invariant((credit_card_df["AMT_DRAWINGS_ATM_CURRENT"].isna() & credit_card_df["AMT_DRAWINGS_CURRENT"] > 0), "AMT_DRAWINGS_ATM_CURRENT is null when AMT_DRAWINGS_CURRENT is bigger than 0",data_frame_size)

check_invariant((credit_card_df["AMT_DRAWINGS_ATM_CURRENT"].isna() & (credit_card_df["AMT_DRAWINGS_POS_CURRENT"].notna() | credit_card_df["AMT_DRAWINGS_OTHER_CURRENT"])),"the missing values of AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT are not aligned",data_frame_size)
#this proof 1- AMT_DRAWINGS_ATM_CURRENT is only null when AMT_DRAWINGS_CURRENT is 0. 2- AMT_DRAWINGS_POS_CURRENT, AMT_DRAWINGS_OTHER_CURRENT and AMT_DRAWINGS_ATM_CURRENT are always null at the same time.
#so the missing values of this 3 columns are actually silents 0.


In [ ]:
#13
#We proceed to check if the semantic have consistent representation in magitudes terms in these columns
check_invariant(credit_card_df["AMT_DRAWINGS_ATM_CURRENT"] > credit_card_df["AMT_DRAWINGS_CURRENT"],"AMT_DRAWINGS_ATM_CURRENT are bigger than AMT_DRAWINGS_CURRENT", data_frame_size)
check_invariant(credit_card_df["AMT_DRAWINGS_POS_CURRENT"] > credit_card_df["AMT_DRAWINGS_CURRENT"],"AMT_DRAWINGS_POS_CURRENT are bigger than AMT_DRAWINGS_CURRENT", data_frame_size)
check_invariant(credit_card_df["AMT_DRAWINGS_OTHER_CURRENT"] > credit_card_df["AMT_DRAWINGS_CURRENT"],"AMT_DRAWINGS_OTHER_CURRENT are bigger than AMT_DRAWINGS_CURRENT", data_frame_size)

summatory= (credit_card_df["AMT_DRAWINGS_ATM_CURRENT"]  + credit_card_df["AMT_DRAWINGS_POS_CURRENT"] + credit_card_df["AMT_DRAWINGS_OTHER_CURRENT"])

columns_summed_mask= summatory == credit_card_df["AMT_DRAWINGS_CURRENT"]
check_invariant(~columns_summed_mask,"the sumn of the 3 columns of DRAWINGS don't are equal to the column that holds the total (AMT_DRAWINGS_CURRENT)", data_frame_size)


check_invariant(summatory > credit_card_df["AMT_DRAWINGS_CURRENT"], "the sum of the 3 column exceed the total (AMT_DRAWINGS_CURRENT)", data_frame_size)


diff= (credit_card_df["AMT_DRAWINGS_OTHER_CURRENT"]  + credit_card_df["AMT_DRAWINGS_POS_CURRENT"] + credit_card_df["AMT_DRAWINGS_OTHER_CURRENT"]) - (credit_card_df["AMT_DRAWINGS_CURRENT"])
diff.describe()

In [ ]:
#14
credit_card_df[credit_card_df["AMT_DRAWINGS_ATM_CURRENT"] > credit_card_df["AMT_DRAWINGS_CURRENT"]].head(10)
credit_card_df[credit_card_df["AMT_DRAWINGS_CURRENT"]  < 0].head()
#The invariant don't are violated for negative values on AMT_DRAWINGS_CURRENT. But we don't have enough cases to modelate this, so we have to discard it clipping in 0 the negative side of the scale.

In [ ]:
#15
min_bigger_than_total_mask= (credit_card_df["AMT_INST_MIN_REGULARITY"] > credit_card_df["AMT_TOTAL_RECEIVABLE"])
check_invariant(min_bigger_than_total_mask, "the expected ammount to be paid is less than the minimun ammount to don't incurre in delincuency" , data_frame_size)

invariant_violated_cases= credit_card_df[min_bigger_than_total_mask]
unexpected_difference= invariant_violated_cases["AMT_INST_MIN_REGULARITY"] - invariant_violated_cases["AMT_TOTAL_RECEIVABLE"]
unexpected_difference.describe()


In [ ]:
#16
credit_card_df["inconsistency_o_penalization"] = (invariant_violated_cases["AMT_INST_MIN_REGULARITY"] - invariant_violated_cases["AMT_TOTAL_RECEIVABLE"]).clip(lower=0)
cases_of_analisis= credit_card_df[credit_card_df["inconsistency_o_penalization"] > 0]
sns.relplot(data= cases_of_analisis, x="AMT_BALANCE", y= "inconsistency_o_penalization", kind="scatter")
#This leads us to several interesting conclusions: first, it reinforces the premise that AMT_BALANCE  needs its lower limit clipped at 0 to avoid artifacts. 
#Negative balance (overpayment) can lead to errors when the metrics are calculated, such as in this case where a negative AMT_TOTAL_RECEIVABLE make AMT_INST_MIN_REGULARITY 
#bigger than the field that represent the rest of the debt 
#mixing them with the cases where more money is being demanded than the total of the remaining loan

In [ ]:
#17
mask_capital_bigger_than_total= (credit_card_df["AMT_RECEIVABLE_PRINCIPAL"] > credit_card_df["AMT_TOTAL_RECEIVABLE"])
print(mask_capital_bigger_than_total.sum())


excluding_overpayment= credit_card_df[(mask_capital_bigger_than_total) & (credit_card_df["AMT_TOTAL_RECEIVABLE"] > 0)]
print(len(excluding_overpayment))

pd.set_option('display.max_columns', None)
excluding_overpayment.head(20)


In [ ]:
#18
credit_card_df["NEXT_STATUS"] = credit_card_df.groupby("SK_ID_PREV")["NAME_CONTRACT_STATUS"].shift(-1)
closed_mask = ((credit_card_df["NEXT_STATUS"] != "Completed") & (credit_card_df["NEXT_STATUS"].notna())) 
reopen_rows=credit_card_df[(credit_card_df["NAME_CONTRACT_STATUS"] == "Completed") & (closed_mask)]
reopen_rows.head(20)

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF,NEXT_STATUS


In [11]:
have_at_least_one_status_closed= credit_card_df["NAME_CONTRACT_STATUS"].eq("Completed").groupby(credit_card_df["SK_ID_PREV"]).transform("any")
have_recent_balance= credit_card_df["MONTHS_BALANCE"].gt(-3).groupby(credit_card_df["SK_ID_PREV"]).transform("any")

is_closed = credit_card_df["NAME_CONTRACT_STATUS"] == "Completed"

credit_card_df["closing_month"] = (credit_card_df["MONTHS_BALANCE"].where(is_closed).groupby(credit_card_df["SK_ID_PREV"]).transform("min"))
credit_card_df["non_closed_loan"] = (~have_at_least_one_status_closed)
credit_card_df["potential_on_going_loan"] = (~have_at_least_one_status_closed) & (have_recent_balance)
credit_card_df["incomplete_sequence"] =  (~have_at_least_one_status_closed) & (~have_recent_balance)

